## QC: Stream Temperature Statistics

Spot-check `wt_stats.nc` by pulling 20 random stream IDs and displaying all statistics in tables. Goal is to verify values are in plausible ranges — not a formal validation.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import random
import sys
sys.path.insert(0, '..')
from luts_wt import wt_stat_var_dict

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Change this path to the sanity-check output while the full run is pending
WT_STATS_PATH = '/beegfs/CMIP6/jdpaul3/arctic_rivers_data/wt_stats.nc'

ds = xr.open_dataset(WT_STATS_PATH)
print(ds)

In [ ]:
random.seed(42)
all_stream_ids = ds.stream_id.values.tolist()
sample_ids = random.sample(all_stream_ids, min(20, len(all_stream_ids)))
print(f'Sampled {len(sample_ids)} stream IDs:', sample_ids)

In [ ]:
def stats_table(ds, var_names, stream_ids, model, era, source='original_gcm'):
    """Return a DataFrame: rows = variables, columns = stream_ids."""
    rows = {}
    for var in var_names:
        units = wt_stat_var_dict[var]['units']
        row_label = f'{var}  [{units}]'
        vals = (
            ds[var]
            .sel(model=model, era=era, source=source)
            .sel(stream_id=stream_ids)
            .values
        )
        rows[row_label] = vals
    return pd.DataFrame(rows, index=stream_ids).T

## Threshold exceedance — days above 13 / 18 / 20 °C

**Expected ranges (historical, Alaska streams):** most streams 0–60 days above 13°C; fewer above 18°C; very few or zero above 20°C for cold headwater streams.

In [ ]:
thresh_vars = ['wt_days_gt13_mean', 'wt_days_gt18_mean', 'wt_days_gt20_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, thresh_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== C2LE2 1990-2021 ===')
display(stats_table(ds, thresh_vars, sample_ids, 'C2LE2', '1990-2021'))

print('\n=== C2LE2 2034-2065 (expect higher counts) ===')
display(stats_table(ds, thresh_vars, sample_ids, 'C2LE2', '2034-2065'))

## Monthly mean temperatures

**Expected pattern:** near 0.1°C in winter (Nov–Mar), warming through spring, peak in July/August, cooling through fall. Should not see values > ~25°C or < 0°C.

In [ ]:
mean_vars = [f'wt_mean_{m}' for m in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']]

print('=== Historical 1990-2021: monthly mean temperatures ===')
display(stats_table(ds, mean_vars, sample_ids, 'historical', '1990-2021'))

## Monthly min and max temperatures

Min should be at or near 0.1°C (model floor). Max should be the warmest average July/August across all years — slightly above the mean.

In [ ]:
min_vars = [f'wt_min_{m}' for m in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']]
max_vars = [f'wt_max_{m}' for m in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']]

print('=== Historical 1990-2021: monthly min temperatures ===')
display(stats_table(ds, min_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== Historical 1990-2021: monthly max temperatures ===')
display(stats_table(ds, max_vars, sample_ids, 'historical', '1990-2021'))

## Annual maximum daily temperature — magnitude and timing

**Magnitude:** should be the highest daily value of the year, above the monthly mean peak. Roughly 10–25°C for most Alaska streams.

**Julian day:** peak should fall in summer — roughly day 150–240 (early June to late August).

In [ ]:
ann_vars = ['wt_ann_max_temp_mean', 'wt_ann_max_temp_doy_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, ann_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== C2LE2 2034-2065 (expect slightly higher temps, similar DOY) ===')
display(stats_table(ds, ann_vars, sample_ids, 'C2LE2', '2034-2065'))

## Maximum 7-day rolling average — magnitude and timing

**Magnitude:** should be slightly lower than the daily max (smoothed). **Julian day:** similar to daily max DOY — the 7-day window centers near the annual peak. Values will differ by a few days from `wt_ann_max_temp_doy_mean`.

In [ ]:
roll_vars = ['wt_7d_max_temp_mean', 'wt_7d_max_temp_doy_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, roll_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== Difference: daily max minus 7-day max (expect small positive values) ===')
diff = (
    ds['wt_ann_max_temp_mean'].sel(model='historical', era='1990-2021', source='original_gcm').sel(stream_id=sample_ids)
    - ds['wt_7d_max_temp_mean'].sel(model='historical', era='1990-2021', source='original_gcm').sel(stream_id=sample_ids)
).to_pandas().rename('daily_max - 7d_max  [degC]')
display(diff.to_frame().T)

## Cumulative degree days above 0°C (May–September)

**Expected range:** all values > 0 (minimum T_stream is 0.1°C × 153 May–Sept days ≈ 15 °C·days floor). Warm lowland streams could reach ~3800 °C·days. Cold headwater streams near the floor.

In [ ]:
cdd_vars = ['wt_cdd_may_sept_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, cdd_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== C2LE2 2034-2065 (expect higher CDD) ===')
display(stats_table(ds, cdd_vars, sample_ids, 'C2LE2', '2034-2065'))

## NaN structure checks

Verify the expected NaN patterns from the `source` dimension:
- `historical` model has no future projections → all-NaN in the `2034-2065` era for `original_gcm`
- `PGWh`/`PGWm` are future-only → all-NaN in the `1990-2021` era for `original_gcm`
- `gcm_diff` past era should always be all-NaN (by design)
- GCM models (C2LE*) should have data in both eras

In [ ]:
sid = sample_ids[0]
check_var = 'wt_ann_max_temp_mean'

checks = {
    'historical / 2034-2065 / original_gcm  (expect NaN)':  ds[check_var].sel(model='historical', era='2034-2065', source='original_gcm', stream_id=sid).item(),
    'PGWh      / 1990-2021 / original_gcm  (expect NaN)':   ds[check_var].sel(model='PGWh',       era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'PGWm      / 1990-2021 / original_gcm  (expect NaN)':   ds[check_var].sel(model='PGWm',       era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'any model / 1990-2021 / gcm_diff      (expect NaN)':   ds[check_var].sel(model='C2LE2',      era='1990-2021', source='gcm_diff',      stream_id=sid).item(),
    'historical / 1990-2021 / original_gcm (expect value)': ds[check_var].sel(model='historical', era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'C2LE2     / 1990-2021 / original_gcm  (expect value)': ds[check_var].sel(model='C2LE2',      era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'C2LE2     / 2034-2065 / original_gcm  (expect value)': ds[check_var].sel(model='C2LE2',      era='2034-2065', source='original_gcm', stream_id=sid).item(),
    'C2LE2     / 2034-2065 / gcm_diff      (expect value)': ds[check_var].sel(model='C2LE2',      era='2034-2065', source='gcm_diff',      stream_id=sid).item(),
}

for label, val in checks.items():
    is_nan = np.isnan(val)
    expected_nan = 'expect NaN' in label
    status = 'OK' if is_nan == expected_nan else 'FAIL'
    print(f'[{status}]  {label}  →  {val}')


## original_gcm vs gcm_diff_applied_to_cheng (2034–2065)

`gcm_diff_applied_to_cheng` applies each GCM's delta (future − past) to the historical Cheng baseline rather than using the raw GCM future value:
```
gcm_diff_applied_to_cheng = historical_past + (gcm_future − gcm_past)
```
This comparison checks how much the two approaches diverge. Large `orig-adj` values mean the GCM and the historical Cheng run disagree on the baseline, so the choice of source matters for the front end. Only GCM models (C2LE*) have non-NaN values for both sources in the future era.


In [ ]:

GCM_MODELS = ['C2LE2', 'C2LE4', 'C2LE7', 'C2LE9']

def comparison_table(ds, var_names, stream_ids, model, era='2034-2065'):
    """Side-by-side comparison of original_gcm vs gcm_diff_applied_to_cheng.
    Returns a DataFrame with three row blocks per variable:
      - original   (raw GCM future)
      - adjusted   (historical Cheng baseline + GCM delta)
      - orig-adj   (difference; positive = GCM runs warmer than bias-corrected)
    """
    rows = {}
    for var in var_names:
        units = wt_stat_var_dict[var]['units']
        orig = ds[var].sel(model=model, era=era, source='original_gcm').sel(stream_id=stream_ids).values
        adj  = ds[var].sel(model=model, era=era, source='gcm_diff_applied_to_cheng').sel(stream_id=stream_ids).values
        rows[f'{var}  original  [{units}]'] = orig
        rows[f'{var}  adjusted  [{units}]'] = adj
        rows[f'{var}  orig-adj  [{units}]'] = orig - adj
    return pd.DataFrame(rows, index=stream_ids).T


In [ ]:

stat_groups = [
    ('Threshold exceedance (days)',        thresh_vars),
    ('Monthly mean temperatures',          mean_vars),
    ('Annual max + 7-day rolling max',     ann_vars + roll_vars),
    ('Cumulative degree days (May–Sept)',   cdd_vars),
]

for model in GCM_MODELS:
    print(f'\n{"="*70}')
    print(f'  MODEL: {model}  |  era: 2034-2065')
    print(f'{"="*70}')
    for group_label, var_list in stat_groups:
        print(f'\n--- {group_label} ---')
        print('  rows: [variable  original], [variable  adjusted], [variable  orig-adj]')
        display(comparison_table(ds, var_list, sample_ids, model))


In [ ]:

# Cross-model summary: orig − adj for annual max temperature
# One row per GCM model, columns are the 20 sampled stream IDs.
# Shows at a glance whether divergence between sources is consistent across models.
summary_rows = {}
for model in GCM_MODELS:
    orig = ds['wt_ann_max_temp_mean'].sel(model=model, era='2034-2065', source='original_gcm').sel(stream_id=sample_ids).values
    adj  = ds['wt_ann_max_temp_mean'].sel(model=model, era='2034-2065', source='gcm_diff_applied_to_cheng').sel(stream_id=sample_ids).values
    summary_rows[model] = orig - adj

summary_df = pd.DataFrame(summary_rows, index=sample_ids).T
summary_df.index.name = 'model'
print('orig − adj for wt_ann_max_temp_mean  [degC]  (2034-2065)')
display(summary_df)
